# Eval: Tool Selector Agent (Contribution 2)

Compares baseline (manager sees all 27 tools) vs filtered (manager sees top-7)
on 15 natural language queries covering all tool categories.

Mirrors the 10-task protocol from the original PhenoAssistant paper.

In [ ]:
import os, json
from functions.generic_tools import set_env_vars
set_env_vars('./.env.yaml')
from openai import AzureOpenAI
from utils.tool_selector import ToolSelectorIndex

client = AzureOpenAI(
    api_key=os.environ['OPENAI_API_KEY'],
    azure_endpoint=os.environ['AZURE_API_URL'],
    api_version=os.environ['AZURE_API_VERSION'],
)
MODEL = os.environ['MODEL_NAME']

# Load the full tool list from agents.py's registered tools
import agents  # this registers everything
ALL_TOOLS = agents.manager.llm_config['tools']
tool_index = agents.tool_index
print(f'Total tools registered: {len(ALL_TOOLS)}')

In [ ]:
# Eval tasks: (query, expected_tool_name)
# Covers all major tool categories including your Contribution 1 models
EVAL_TASKS = [
    # Classification
    ('Classify rice leaf images for nitrogen deficiency severity',
     'infer_image_classification'),
    ('Identify nutrient stress in banana leaves from close-range RGB photos',
     'infer_image_classification'),
    ('Classify wheat drone images for nutrient deficiency',
     'infer_image_classification'),
    ('Detect nutrient deficiency in maize corn leaf images',
     'infer_image_classification'),
    # Detection / counting
    ('Count wheat spikes in field images',
     'infer_object_detection'),
    ('Detect and count rice panicles from UAV imagery',
     'infer_object_detection'),
    # Segmentation
    ('Segment individual arabidopsis leaves and count them',
     'infer_instance_segmentation'),
    ('Segment potato leaves from close-range RGB images',
     'infer_instance_segmentation'),
    # Regression
    ('Predict plant height from canopy images',
     'infer_image_regression'),
    ('Estimate leaf area index from field photos',
     'infer_image_regression'),
    # Temporal / satellite
    ('Analyse Sentinel-2 time series for crop type mapping and harvest prediction',
     'infer_temporal_analysis'),
    # Fine-tuning
    ('Train a new segmentation model on my uploaded arabidopsis dataset',
     'finetune_instance_segmentation'),
    # Coding
    ('Plot a bar chart comparing deficiency class counts across crops',
     'coding'),
    # Synonym / vague
    ('My paddy leaves look yellow, run deficiency classification',
     'infer_image_classification'),
    ('Phenotype rosette leaves and measure their area',
     'infer_instance_segmentation'),
]
print(f'Eval tasks: {len(EVAL_TASKS)}')

In [ ]:
def ask_manager_to_pick_tool(query, tools):
    """Ask the LLM to pick the best tool from a given list."""
    tool_list = '\n'.join(
        f"- {t['function']['name']}: {t['function'].get('description','')[:80]}"
        for t in tools
    )
    resp = client.chat.completions.create(
        model=MODEL,
        messages=[{
            'role': 'user',
            'content': (
                f'Task: "{query}"\n\n'
                f'Available tools:\n{tool_list}\n\n'
                'Reply with ONLY the single tool name that best fits this task. '
                'Nothing else — just the tool name.'
            )
        }],
        temperature=0,
        max_tokens=20,
    )
    return resp.choices[0].message.content.strip()

print('Running BASELINE (all 27 tools shown to manager)...')
baseline_results = []
for query, expected in EVAL_TASKS:
    predicted = ask_manager_to_pick_tool(query, ALL_TOOLS)
    correct = expected in predicted
    baseline_results.append(correct)
    print(f'  [{"PASS" if correct else "FAIL"}] {query[:55]}')
    if not correct:
        print(f'         expected={expected}, got={predicted}')

baseline_acc = sum(baseline_results) / len(baseline_results)
print(f'\nBaseline accuracy: {baseline_acc:.0%} ({sum(baseline_results)}/{len(baseline_results)})')

In [ ]:
print('Running CONTRIBUTION 2 (top-7 filtered tools shown to manager)...')
filtered_results = []
retrieval_recall = []  # was the right tool even in the top-7?

for query, expected in EVAL_TASKS:
    top_k = tool_index.get_top_k(query, k=7)
    top_k_names = [t['function']['name'] for t in top_k]
    in_top_k = expected in top_k_names
    retrieval_recall.append(in_top_k)

    if in_top_k:
        predicted = ask_manager_to_pick_tool(query, top_k)
        correct = expected in predicted
    else:
        predicted = '[not retrieved]'
        correct = False

    filtered_results.append(correct)
    status = 'PASS' if correct else ('MISS' if not in_top_k else 'FAIL')
    print(f'  [{status}] {query[:55]}')
    if not correct:
        print(f'         expected={expected}, got={predicted}, in_top7={in_top_k}')

filtered_acc = sum(filtered_results) / len(filtered_results)
recall = sum(retrieval_recall) / len(retrieval_recall)
print(f'\nRetrieval recall (right tool in top-7): {recall:.0%}')
print(f'Filtered accuracy:                      {filtered_acc:.0%} ({sum(filtered_results)}/{len(filtered_results)})')

In [ ]:
import pandas as pd
rows = []
for i, (query, expected) in enumerate(EVAL_TASKS):
    rows.append({
        'query': query,
        'expected_tool': expected,
        'baseline_correct': baseline_results[i],
        'retrieval_recall': retrieval_recall[i],
        'filtered_correct': filtered_results[i],
    })
df = pd.DataFrame(rows)
print(df[['query','baseline_correct','retrieval_recall','filtered_correct']].to_string(index=False))
print(f'\nBaseline accuracy:  {baseline_acc:.0%}')
print(f'Retrieval recall:   {recall:.0%}')
print(f'Filtered accuracy:  {filtered_acc:.0%}')
print(f'Improvement:        +{(filtered_acc - baseline_acc):.0%}')
os.makedirs('results', exist_ok=True)
df.to_csv('results/eval_tool_selector.csv', index=False)
print('Saved to results/eval_tool_selector.csv')